# Sociolinguistics : observing linguistic variation in social context

In this Assignment we want to focus on a task you might be interested in if you are studying sociolinguistics or social variation in language use. We focus on language variation in relation to specific identity marker: age. We already manually annotated some data and found some potential lexical choices or grammatical constructions that might be more frequent in younger versus older speakers. 

To have some data to study the differences in language between people of younger versus older age groups we need texts with speaker information. As an approximation we use two different Reddit communities (sub-reddits) that are focused on different age groups:
- r/GenZ (a community for members of Generation Z)
- r/AskOldPeople (a community for older adults to share their experiences and advice)

Of course we cannot be sure that all members of these communities belong to the respective age groups, but we can assume that there will be a lot of overlap.

The goal of this assignment is to find out differences in language use between these two communities.

As practical methods we want to learn:
- how to add additional information to an existing corpus object
- how to extract linguistic features that are useful to study language variation
- how to compare the two communities based on these features

## Part 1 of the Assignment:

Before we can do the analysis we will conduct a few pre-processing steps. We will focus on this part first and then do the analysis.
(1) Tokenization of the corpus
(2) Esxtract different features
(3) Create new corpus objects that contain utterances and the new type of information.

## Loading the Reddit Corpus in Convokit
The Convokit library provides a ready-to-use Reddit corpus that we can use for our analysis. Instead of loading the full corpus, we can load specific sub-reddits. For our analysis we want to load the two sub-reddits mentioned above: r/GenZ and r/AskOldPeople.

*Task*: Load the two different datasets and compare the size of the two corpora (number of speakers and number of utterances).
Add tokenization as a pre-processing, since we want to use number of tokens as information to filter out reddit posts that are very short or very long. Use the TextParser preprocessor for this and add this transformation to both corpora. You might need to manually download nltk resources if you get an error (nltk.download('punkt_tab')). The tokenization might take a while depending on your machine (it took ~7min on my laptop for both corpora).


In [ ]:
from convokit import Corpus, download

corpus_old = Corpus(filename=download("subreddit-AskOldPeople"))
corpus_young = Corpus(filename=download("subreddit-GenZ"))


In [ ]:
print("Old People Subreddit:")
print(corpus_old.print_summary_stats())
print("\nGen Z Subreddit:")
print(corpus_young.print_summary_stats())

In [ ]:
import nltk

nltk.download('punkt_tab')
from convokit.text_processing import TextParser

# create a tokenizer and apply it to the corpora
tokenizer = TextParser(mode="tokenize")
corpus_old_tokenized = tokenizer.transform(corpus_old)
corpus_young_tokenized = tokenizer.transform(corpus_young)

## Filter out very short and very long posts
After pre-processing the tokens should be stored in the meta.parsed field.
 
*Task*: Add another meta information field 'num_tokens' that contains the number of tokens in each post. You can iterate over the utterances in each corpus and count the number of tokens based on the meta.parsed field. Add a new metadata field to each utterance object ('num_tokens') that contains the number of tokens. Then filter out utterances that have less than 11 tokens or more than 349 tokens and store the filtered utterances in a dataframe. How much data is left after filtering?

In [ ]:
# iterate over utterance objects and create a new meta field 'num_tokens'
for utt in corpus_old_tokenized.iter_utterances():
    sentences = utt.meta['parsed']
    num_tokens = sum(len(sent['toks']) for sent in sentences)
    utt.meta['num_tokens'] = num_tokens
for utt in corpus_young_tokenized.iter_utterances():
    sentences = utt.meta['parsed']
    num_tokens = sum(len(sent['toks']) for sent in sentences)
    utt.meta['num_tokens'] = num_tokens

In [ ]:
# filter utterances based on num_tokens and store in a dataframe
utterances_genz = corpus_young_tokenized.get_utterances_dataframe()
filtered_utterances_genz = utterances_genz[
    (utterances_genz['meta.num_tokens'] > 10) & (utterances_genz['meta.num_tokens'] < 350)]
utterances_old = corpus_old_tokenized.get_utterances_dataframe()
filtered_utterances_old = utterances_old[
    (utterances_old['meta.num_tokens'] > 10) & (utterances_old['meta.num_tokens'] < 350)]
print(f"Gen Z corpus: {len(filtered_utterances_genz)} utterances left after filtering.")
print(f"Ask Old People corpus: {len(filtered_utterances_old)} utterances left after filtering.")

In [ ]:
filtered_utterances_genz.head()

## Create a new corpus object with filtered utterances.

We now want to create a new corpus object in which we want to store all our relevant information for the analysis. The new corpus objects should only contain the filtered utterances. You can create a new corpus object in Convokit by providing utterance objects to the Corpus constructor:
Corpus(utterances=list_of_utterance_objects).

*Task*: Create two new corpus objects (one for each age group) that only contain the filtered utterances.

In [ ]:
utterance_objects_genz = []
for utt_id in filtered_utterances_genz.index:
    utt = corpus_young_tokenized.get_utterance(utt_id)
    utterance_objects_genz.append(utt)
corpus_genz_filtered = Corpus(utterances=utterance_objects_genz)
utterance_objects_old = []
for utt_id in filtered_utterances_old.index:
    utt = corpus_old_tokenized.get_utterance(utt_id)
    utterance_objects_old.append(utt)
corpus_old_filtered = Corpus(utterances=utterance_objects_old)

## Extract linguistic features
Now we want to extract some linguistic features that we can use to compare the two age groups. We will focus on two types of features:
- **Lexical features:** We will extract the frequency of specific expressions that might be distinctive for the Gen Z generation (e.g., slang words or specific phrases).
- **Syntactic features:** We will extract the frequency of specific POS tags (e.g. n_adj is the amount of adjetives in an utterance) and for each POS tag how many unique words the utterance contains (e.g. maybe a person uses a lot of adjectives but always the same one). The unique POS tags thus measure lexical variation within a certain syntactic category. 
- **linguistic complexity:** we use corrected type token ratio (CTTR) as a measure of lexical diversity (complexity) in an utterance. CTTR is calculated as the number of unique tokens divided by the square root of two times the total number of tokens. The higher the CTTR, the more diverse the vocabulary used in the utterance. We look at the average number of words per sentence to get an idea of whether a speaker uses longer or shorter sentences on average. And we look at the average age of aquisition for the words used in the utterance (based on a predefined lexicon that contains age of aquisition ratings for words). Higher values indicate that the speaker uses words that are typically learned later in life, which can be an indicator of linguistic sophistication.

## Lexical features
I provided a word list of Gen Z unigrams / n-grams that you can use to extract lexical features. The file is called 'genz_wordlist.txt' and contains one word or n-gram per line. Feel free to add more words that you think are relevant for the analysis. 
*Task*: Load the word list and extract the total number of Gen-Z words used in each utterance. Store this information in a new meta field 'meta.num_genz_words' for each utterance.

In [1]:
# load genz wordlist
with open("/Users/falkne/PycharmProjects/nlpforSocialInteractions/additional_data/gen_z_wordlist.txt", "r") as f:
    genz_wordlist = f.read().splitlines()
print("First 10 words in the genz wordlist:", genz_wordlist[:10])

First 10 words in the genz wordlist: ['queen', 'based', "she's not wrong", 'alt', 'fake news', 'caught up fr', 'fanum tax', 'big cap', 'dub', 'boutta']


In [2]:
# function to count genz expressions in an utterance
def count_gen_z_words(utterance):
    tokens = utterance.meta['parsed'][0]['toks']
    tokens = [t["tok"].lower() for t in tokens]
    count_unigrams = sum(1 for t in tokens if t in genz_wordlist)
    total_freq = 0
    for expression in genz_wordlist:
        if " " in expression:
            freq = utterance.text.lower().count(expression.lower())
            total_freq += freq
    total_freq+=count_unigrams
    return total_freq

In [ ]:
for utt in corpus_genz_filtered.iter_utterances():
    num_genz_words = count_gen_z_words(utt)
    utt.meta['num_genz_words'] = num_genz_words
for utt in corpus_old_filtered.iter_utterances():
    num_genz_words = count_gen_z_words(utt)
    utt.meta['num_genz_words'] = num_genz_words

In [ ]:
# check if now an utterance has the new meta field
sample_utt = next(corpus_genz_filtered.iter_utterances())
print("Sample utterance meta information:", sample_utt.meta)

## Syntactic features and linguistic complexity

To extract these features we use an existing library. The library is called *LFTK* : https://lftk.readthedocs.io/en/latest/
It provides a list of features that can be extracted from text, including POS tag frequencies and measures of lexical diversity.
 
As a first step we will extract the feature names of the features we want to look at. One can use the "search_features" function to search for features based on domain and family. a in a_ stands for average, and n_ stands for number of occurrences. ps stands for per sentence.

In [ ]:
import lftk

pos_features = lftk.search_features(domain='syntax', family="partofspeech", language="general",
                                    return_format="list_dict")
pos_features = [f['key'] for f in pos_features]
additional_features = ["a_word_ps", "a_bry_ps", "corr_ttr"]
features_to_extract = pos_features + additional_features
print(features_to_extract)

## Extract features 

To see how to extract features we will run a small text on a very small portion of the data (extracting the features for the full datasets will take a while (around 2-3 hours). 

*Task*:
(1) Retrieve the utterance texts for the first 100 utterances in each corpus and store them in a list.
(2) use spacy and a small model to process the utterances with the .pipe function (this is faster than processing each utterance individually). The result should be a list of processed spacy documents.
(3) initialize an extractor for each corpus with the list of processed spacy documents. use the extractor.extract(features=features_to_extract) function to extract the features for the processed utterances. The result is a list of dictionaries (one dictionary per utterance). Each dictionary contains the feature values for the respective utterance.
(4) print the first utterance to see how it looks like.




In [ ]:
utterance_samples_genz = corpus_genz_filtered.get_utterances_dataframe().iloc[:100]['text'].tolist()
utterance_samples_old = corpus_old_filtered.get_utterances_dataframe().iloc[:100]['text'].tolist()

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
# process utterances with spacy pipe
processed_utterances_genz = list(nlp.pipe(utterance_samples_genz))
processed_utterances_old = list(nlp.pipe(utterance_samples_old))

In [ ]:
# extract features with lftk
import lftk
LFTK = lftk.Extractor(docs=processed_utterances_genz)
print("LFTK initialized for Gen Z utterances.")
features_genz_sample = LFTK.extract(features=features_to_extract)
print("Extracted features for Gen Z utterances:")
LFTK_old = lftk.Extractor(docs=processed_utterances_old)
print("LFTK initialized for Old People utterances.")
features_old_sample = LFTK_old.extract(features=features_to_extract)
print("Extracted features for Old People utterances:")

In [ ]:
print("Gen Z features sample:")
print(features_genz_sample[0])

## Full Features

Since it takes too long for you to do the feature extraction for the assignment, I already ran the feature extraction for the full datasets and stored the results in two csv files in 'additional_data':

- features_genz.csv
- features_askoldpeople.csv

The files have the size of the utterance dataframes of each corpus after filtering - one row per utterance. Each column is a linguistic feature. Since we want to compare the two corpora based on these features, we normalize the raw counts:
- for n_pos and num_genz_words we divide by the number of tokens in the utterance (meta.num_tokens)
- for u_pos we divide by the number of corresponding n_pos feature (e.g. u_noun / n_noun) (to get the proportion of unique nouns out of all nouns used in the utterance)
- for the other features (a_word_ps, a_bry_ps, corr_ttr) no normalization is needed since they are already averages or ratios.

*Task*: Load the two csv files into pandas dataframes. Set the index to the same index as the utterance dataframes so that each utterance in the feature dataframes corresponds to the same utterance in the utterance dataframes. Add the features as meta information to the respective utterance objects in the two corpora. (You can iterate over the utterance objects, retrieve the utterance id, and then get the feature values from the dataframe based on the index). Normalize the feature values as described above.

In [ ]:
import pandas as pd
features_genz = pd.read_csv("/Users/falkne/PycharmProjects/nlpforSocialInteractions/additional_data/features_genz.csv")
features_old = pd.read_csv("/Users/falkne/PycharmProjects/nlpforSocialInteractions/additional_data/features_askoldpeople.csv")

features_genz.index = corpus_genz_filtered.get_utterances_dataframe().index
features_old.index = corpus_old_filtered.get_utterances_dataframe().index

In [ ]:
# add features to utterance meta information
for utt in corpus_genz_filtered.iter_utterances():
    utt_id = utt.id
    features_utterance = features_genz.loc[utt_id]
    
    for feature in features_genz.columns:
        value = features_utterance[feature]
        # normalize if needed
        if feature.startswith("n_") and not feature.startswith("n_u"):
            value = value / utt.meta['num_tokens']
        elif feature == "num_genz_words":
            value = value / utt.meta['num_tokens']
        elif feature.startswith("n_u"):
            corresponding_n_feature = feature.replace("n_u", "n_")
            n_value = features_utterance[corresponding_n_feature]
            if n_value > 0:
                value = value / n_value
            else:
                value = 0.0
        utt.meta[feature] = value

In [0]:
features_genz.head()

In [0]:
print(features_genz.columns)
print(len(features_genz.columns))

In [ ]:
for utt in corpus_old_filtered.iter_utterances():
    utt_id = utt.id
    features_utterance = features_old.loc[utt_id]
    
    for feature in features_old.columns:
        value = features_utterance[feature]
        # normalize if needed
        if feature.startswith("n_") and not feature.startswith("n_u"):
            value = value / utt.meta['num_tokens']
        elif feature.startswith("n_u"):
            corresponding_n_feature = feature.replace("n_u", "n_")
            n_value = features_utterance[corresponding_n_feature]
            if n_value > 0:
                value = value / n_value
            else:
                value = 0.0
        utt.meta[feature] = value

In [ ]:
# check the meta data of a sample utterance
random_utt = corpus_genz_filtered.random_utterance()
print("Sample utterance meta information after adding features:", random_utt.meta)